In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException, StaleElementReferenceException
from selenium.webdriver.support.ui import Select
import time
import pandas as pd
from datetime import datetime, timedelta
import re

In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

options = webdriver.ChromeOptions()
options.add_argument(f"user-agent={user_agent}")

service = Service("C:/Program Files (x86)/chromedriver-win64/chromedriver.exe")

driver = webdriver.Chrome(service=service, options=options)

In [ ]:
# call for tenders notices

link = "LINK" #add link

driver.get(link)
time.sleep(3)

In [ ]:
# scraping

deadline_date = datetime.now() - timedelta(days=30)
stop_scraping = False

tenders= []

def scrape_current_page():
    global stop_scraping
    rows = driver.find_elements(By.CSS_SELECTOR, "#DataTables_Table_0 tbody tr")
    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 4:
            title = cells[2].text.strip().encode('utf-8', errors='ignore').decode('utf-8')
            publish_date = cells[3].text.strip()

            try:
                date_obj = datetime.strptime(publish_date, "%d/%m/%Y")
            except:
                print("Invalid date:", publish_date)
                continue

            if date_obj < deadline_date:
                print(f"Stopped: date too old → {publish_date}")
                stop_scraping = True
                return

            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": None,
                "link": link,
                "country": "Mali"
            })

while True:
    scrape_current_page()
    if stop_scraping:
        break
    try:
        next_button = driver.find_element(By.ID, "DataTables_Table_0_next")

        driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(2)
    except Exception as e:
        print("Pagination error:", e)
        break


In [ ]:
# expression of interest notices

link ="https://www.dgmp.gouv.ml/#quicktabs-accueil=2"

driver.get(link)
time.sleep(3)

In [ ]:
# scraping

deadline_date = datetime.now() - timedelta(days=30)
stop_scraping = False

def scrape_current_page():
    global stop_scraping
    rows = driver.find_elements(By.CSS_SELECTOR, "#DataTables_Table_2 tbody tr")
    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if len(cells) >= 4:
            title = cells[2].text.strip().encode('utf-8', errors='ignore').decode('utf-8')
            publish_date = cells[3].text.strip()

            try:
                date_obj = datetime.strptime(publish_date, "%d/%m/%Y")
            except:
                print("Invalid date:", publish_date)
                continue

            if date_obj < deadline_date:
                print(f"Stopped: date too old → {publish_date}")
                stop_scraping = True
                return

            tenders.append({
                "title": title,
                "publication_date": publish_date,
                "expiration_date": None,
                "link": link,
                "country": "Mali"
            })

while True:
    scrape_current_page()
    if stop_scraping:
        break
    try:
        next_button = driver.find_element(By.ID, "DataTables_Table_2_next")

        driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", next_button)
        time.sleep(2)
    except Exception as e:
        print("Pagination error:", e)
        break


In [ ]:
driver.quit()

In [ ]:
def clean_excel_string(s):
    if isinstance(s, str):
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

df = pd.DataFrame(tenders)
df = df.applymap(clean_excel_string)

df.to_excel("../2_scraped_files/Mali2.xlsx", index=False) # modify